In [2]:
import pandas as pd
import numpy as np

fy2011 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/H-1B_iCert_LCA_FY2011_Q4.xlsx"

df = pd.read_excel(fy2011)

print(df.shape)
print(df.head())

(358857, 35)
      LCA_CASE_NUMBER               STATUS LCA_CASE_SUBMIT DECISION_DATE  \
0  I-200-09124-371007            CERTIFIED      2011-08-03    2011-08-09   
1  I-200-09125-717448            CERTIFIED      2011-01-11    2011-01-18   
2  I-200-09126-530125  CERTIFIED-WITHDRAWN      2011-04-21    2011-04-27   
3  I-200-09127-370154            CERTIFIED      2011-04-19    2011-04-25   
4  I-200-09127-334015            CERTIFIED      2011-04-04    2011-04-08   

  VISA_CLASS LCA_CASE_EMPLOYMENT_START_DATE LCA_CASE_EMPLOYMENT_END_DATE  \
0       H-1B                     2011-10-01                   2014-09-30   
1       H-1B                     2011-02-01                   2014-02-01   
2       H-1B                     2011-09-01                   2014-09-01   
3       H-1B                     2011-10-01                   2014-10-01   
4       H-1B                     2011-04-04                   2012-03-02   

          LCA_CASE_EMPLOYER_NAME    LCA_CASE_EMPLOYER_ADDRESS  \
0   WILL

In [4]:
print(df.columns.tolist())

['LCA_CASE_NUMBER', 'STATUS', 'LCA_CASE_SUBMIT', 'DECISION_DATE', 'VISA_CLASS', 'LCA_CASE_EMPLOYMENT_START_DATE', 'LCA_CASE_EMPLOYMENT_END_DATE', 'LCA_CASE_EMPLOYER_NAME', 'LCA_CASE_EMPLOYER_ADDRESS', 'LCA_CASE_EMPLOYER_CITY', 'LCA_CASE_EMPLOYER_STATE', 'LCA_CASE_EMPLOYER_POSTAL_CODE', 'LCA_CASE_SOC_CODE', 'LCA_CASE_SOC_NAME', 'LCA_CASE_JOB_TITLE', 'LCA_CASE_WAGE_RATE_FROM', 'LCA_CASE_WAGE_RATE_TO', 'LCA_CASE_WAGE_RATE_UNIT', 'FULL_TIME_POS', 'TOTAL_WORKERS', 'LCA_CASE_WORKLOC1_CITY', 'LCA_CASE_WORKLOC1_STATE', 'PW_1', 'PW_UNIT_1', 'PW_SOURCE_1', 'OTHER_WAGE_SOURCE_1', 'YR_SOURCE_PUB_1', 'LCA_CASE_WORKLOC2_CITY', 'LCA_CASE_WORKLOC2_STATE', 'PW_2', 'PW_UNIT_2', 'PW_SOURCE_2', 'OTHER_WAGE_SOURCE_2', 'YR_SOURCE_PUB_2', 'LCA_CASE_NAICS_CODE']


In [6]:
import pandas as pd

df = pd.read_excel(fy2011)

df["LCA_CASE_EMPLOYER_NAME"] = (
    df["LCA_CASE_EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["LCA_CASE_EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "LCA_CASE_EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

LCA_CASE_EMPLOYER_NAME
MICROSOFT CORPORATION                                 4253
GOOGLE INC.                                           1306
APPLE INC.                                             771
AMAZON CORPORATE LLC                                   624
MICRON TECHNOLOGY, INC.                                391
ADVANCED MICRO DEVICES, INC.                           228
FACEBOOK, INC.                                         222
NVIDIA CORPORATION                                     166
TESLA MOTORS, INC.                                      94
APPLE INC                                               32
AMAZON FULFILLMENT SERVICES, INC.                       28
AMAZON WEB SERVICES LLC                                 25
KENNAMETAL, INC.                                        14
METASENSE, INC.                                         13
AMAZON SERVICES LLC                                     12
MICRONESIAN TELECOMMUNICATIONS CORPORATION              10
AMAZING APPLE, INC.              

In [8]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["LCA_CASE_EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Microsoft    4253
Apple         771
Micron        391
AMD           228
Meta          222
NVIDIA        166
Tesla          94
Name: count, dtype: int64


In [10]:
df_clean = df.copy()

In [12]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["LCA_CASE_WORKLOC1_CITY"] = (
    df["LCA_CASE_WORKLOC1_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["LCA_CASE_WORKLOC1_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Austin      124
New York     26
San Jose     15
Seattle      14
Name: count, dtype: int64


In [14]:
print(df["LCA_CASE_WAGE_RATE_UNIT"].value_counts())

LCA_CASE_WAGE_RATE_UNIT
Year    178
Hour      1
Name: count, dtype: int64


In [16]:
import pandas as pd

# Convert wage values to numbers
df["LCA_CASE_WAGE_RATE_FROM"] = pd.to_numeric(
    df["LCA_CASE_WAGE_RATE_FROM"],
    errors="coerce"
)

# Standardize wage units
df["LCA_CASE_WAGE_RATE_UNIT"] = (
    df["LCA_CASE_WAGE_RATE_UNIT"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["LCA_CASE_WAGE_RATE_FROM"]
    * df["LCA_CASE_WAGE_RATE_UNIT"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_UNIT",
    "Annual_Wage"
]].head(10))

         Company      City  LCA_CASE_WAGE_RATE_FROM LCA_CASE_WAGE_RATE_UNIT  \
721       Micron  San Jose                 130000.0                    YEAR   
2978         AMD    Austin                  84344.0                    YEAR   
4541       Apple    Austin                 102752.0                    YEAR   
4981         AMD    Austin                  71282.0                    YEAR   
7242   Microsoft  New York                 114000.0                    YEAR   
7745   Microsoft  New York                 132000.0                    YEAR   
11159  Microsoft  New York                 170000.0                    YEAR   
11367        AMD    Austin                  71282.0                    YEAR   
13634        AMD    Austin                  71282.0                    YEAR   
27654  Microsoft  New York                  90210.0                    YEAR   

       Annual_Wage  
721       130000.0  
2978       84344.0  
4541      102752.0  
4981       71282.0  
7242      114000.0  
7745

In [18]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count       179.000000
mean      92700.157542
std       21965.457817
min       49660.000000
25%       74500.000000
50%       89280.000000
75%      103885.000000
max      170000.000000
Name: Annual_Wage, dtype: float64


In [20]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [22]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage      Mean_Wage
0     Austin        AMD                104      77600.0   80681.807692
1     Austin      Apple                  5      85000.0   87582.400000
2     Austin  Microsoft                  1     102900.0  102900.000000
3     Austin     NVIDIA                 14      90000.0  100342.957143
4   New York        AMD                  1      95492.8   95492.800000
5   New York      Apple                  1      80000.0   80000.000000
6   New York       Meta                  2     150000.0  150000.000000
7   New York  Microsoft                 22     112925.0  110509.045455
8   San Jose     Micron                 15     115000.0  113674.333333
9    Seattle       Meta                 12     115000.0  119166.666667
10   Seattle  Microsoft                  2     107500.0  107500.000000


In [24]:
columns_to_keep = [
    "Company",
    "LCA_CASE_EMPLOYER_NAME",
    "LCA_CASE_JOB_TITLE",
    "LCA_CASE_SOC_NAME",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_UNIT",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2011.csv",
    index=False
)